####Table Constraints: NOT NULL, CHECK, and Identity Columns
> Schema enforcement works at the structural level. 
Constraints go one level deeper — they enforce business rules at the value level.

In [0]:
USE CATALOG workspace;
USE SCHEMA default;

#### Part 1 — NOT NULL and CHECK Constraints


#####1. Create orders with NOT NULL on customer_id

In [0]:
CREATE OR REPLACE TABLE orders (
  order_id    INT          COMMENT 'Unique order identifier',
  customer_id INT NOT NULL COMMENT 'Must reference a valid customer',
  amount      DOUBLE       COMMENT 'Order total in USD',
  status      STRING       COMMENT 'Order lifecycle status',
  created_at  TIMESTAMP    COMMENT 'Order creation timestamp'
)
USING DELTA
COMMENT 'Order management — constraints demo';

#####2. Valid rows — all constraints satisfied

In [0]:
INSERT INTO orders VALUES
  (1, 101, 149.99, 'pending',    '2024-08-01 10:00:00'),
  (2, 102, 320.00, 'completed',  '2024-08-01 10:05:00'),
  (3, 103, 89.50,  'processing', '2024-08-01 10:10:00');

#####3. [EXPECT ERROR] NULL customer_id violates NOT NULL

In [0]:
INSERT INTO orders VALUES
  (4, NULL, 55.00, 'pending', '2024-08-01 11:00:00');

#####4. Add named CHECK constraint — amount must be positive

In [0]:
ALTER TABLE orders
ADD CONSTRAINT valid_amount CHECK (amount > 0);

#####5. [EXPECT ERROR] Negative amount violates valid_amount

In [0]:
INSERT INTO orders VALUES
  (5, 104, -50.00, 'pending', '2024-08-01 11:05:00');

#####6. Add named CHECK constraint — enumerate allowed status values

In [0]:
ALTER TABLE orders
ADD CONSTRAINT valid_status
CHECK (status IN ('pending', 'processing', 'completed', 'cancelled'));

#####7. [EXPECT ERROR] 'shipped' not in allowed status list

In [0]:
INSERT INTO orders VALUES
  (6, 105, 210.00, 'shipped', '2024-08-01 11:10:00');

#####8. Inspect all constraints — stored as table properties

In [0]:
SHOW TBLPROPERTIES orders;

#####9  Drop valid_status constraint

In [0]:
ALTER TABLE orders DROP CONSTRAINT valid_status;

#####10. Confirm only valid_amount remains

In [0]:
SHOW TBLPROPERTIES orders;

>Constraints are enforced on new writes only. If I add a CHECK constraint to a table that already has rows violating it — those existing rows stay in the table. Delta does not retroactively validate data. 

#### Part 2 — Identity Columns

#####1. Dimension table with GENERATED ALWAYS AS IDENTITY

In [0]:
CREATE OR REPLACE TABLE customers_dim (
  customer_sk  BIGINT GENERATED ALWAYS AS IDENTITY
               COMMENT 'Surrogate key — Delta-managed, always auto-generated',
  customer_id  STRING  NOT NULL COMMENT 'Business key from source system',
  name         STRING  NOT NULL,
  email        STRING,
  tier         STRING,
  is_current   BOOLEAN DEFAULT true
)
USING DELTA
COMMENT 'Customer dimension — star schema surrogate key demo'
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

#####2. Insert WITHOUT specifying customer_sk — Delta generates it

In [0]:
INSERT INTO customers_dim (customer_id, name, email, tier, is_current) VALUES
  ('C001', 'Alice Nguyen',  'alice@example.com',  'gold',     true),
  ('C002', 'Bob Patel',     'bob@example.com',    'silver',   true),
  ('C003', 'Carol Santos',  'carol@example.com',  'platinum', true);

#####3. Verify auto-generated surrogate keys

In [0]:
SELECT customer_sk, customer_id, name, tier
FROM   customers_dim
ORDER  BY customer_sk;

#####4. [EXPECT ERROR] Cannot provide explicit value for ALWAYS GENERATED column

In [0]:
INSERT INTO customers_dim
  (customer_sk, customer_id, name, email, tier, is_current)
VALUES
  (99, 'C004', 'David Kim', 'david@example.com', 'bronze', true);

#####5. BY DEFAULT version — allows explicit key insertion

In [0]:
CREATE OR REPLACE TABLE customers_dim_v2 (
  customer_sk  BIGINT GENERATED BY DEFAULT AS IDENTITY,
  customer_id  STRING NOT NULL,
  name         STRING NOT NULL,
  email        STRING,
  tier         STRING
)
USING DELTA
COMMENT 'Customer dim v2 — BY DEFAULT for migration scenarios';

#####6. Insert WITH explicit customer_sk — migration pattern

In [0]:
INSERT INTO customers_dim_v2
  (customer_sk, customer_id, name, email, tier)
VALUES
  (5001, 'C001', 'Alice Nguyen',  'alice@example.com',  'gold'),
  (5002, 'C002', 'Bob Patel',     'bob@example.com',    'silver');

#####7. Insert WITHOUT customer_sk — Delta generates the next value

In [0]:
INSERT INTO customers_dim_v2 (customer_id, name, email, tier)
VALUES ('C003', 'Carol Santos', 'carol@example.com', 'platinum');

#####8. Show mixed explicit + auto-generated keys

In [0]:
SELECT customer_sk, customer_id, name
FROM   customers_dim_v2
ORDER  BY customer_sk;

####Summary
Delta as a data quality layer at the table level.

NOT NULL. Declare it in the column definition. Delta enforces it on every write path — SQL INSERT, DataFrame append, MERGE. No exceptions.

CHECK. Add it with ALTER TABLE ... ADD CONSTRAINT name CHECK (expression). Any boolean expression works — range checks, IN lists, cross-column rules. Always name your constraints — the name appears in error messages and in SHOW TBLPROPERTIES. Enforced on new writes only; existing rows are not validated. Add constraints at table creation time, not after bad data has already landed.

Identity Columns. GENERATED ALWAYS AS IDENTITY for dimension tables where Delta fully manages the surrogate key — no explicit inserts allowed. GENERATED BY DEFAULT AS IDENTITY for migration scenarios where you need to preserve existing keys. Gaps are normal after deletes — uniqueness is guaranteed, contiguity is not.